In [0]:
import pyspark.sql.functions as F
import plotly.express as px
import pandas as pd
import urllib.request
import json

dbutils.widgets.text("gold_catalog", "dbr_dev")
dbutils.widgets.text("gold_schema", "artemzharkov10_gold")

GOLD_CATALOG = dbutils.widgets.get("gold_catalog")
GOLD_SCHEMA = dbutils.widgets.get("gold_schema")

In [0]:

GOLD_TABLE = f"{GOLD_CATALOG}.{GOLD_SCHEMA}.gold_history_weather_weights"

df_gold = spark.table(GOLD_TABLE)

# 2. Объединение таблиц для получения voivodeship, time и normalized_risk_index
df_joined = df_gold.select("voivodeship", "time", "normalized_risk_index")

# 3. Фильтрация данных
# Вывод всех лет по часам вызовет OutOfMemory на драйвере кластера Databricks. 
# Требуется ограничить выборку (например, одним днем или неделей).
df_filtered = df_joined.filter(F.col("time").between("2024-01-01 00:00:00", "2024-01-03 00:00:00"))

# Перевод в Pandas DataFrame
df_pd = df_filtered.toPandas()

# Округление времени до часа для корректного шага анимации
df_pd['time_str'] = pd.to_datetime(df_pd['time']).dt.strftime('%Y-%m-%d %H:00')
df_pd = df_pd.sort_values('time_str')

In [0]:
# Перевод в Pandas DataFrame
df_pd = df_filtered.toPandas()

# Округление времени до часа
df_pd['time_str'] = pd.to_datetime(df_pd['time']).dt.strftime('%Y-%m-%d %H:00')
df_pd = df_pd.sort_values('time_str')

# Словарь для приведения английских названий или названий с заглавной буквы 
# к точному формату GeoJSON (строчные польские)
geojson_mapping = {
    "West Pomeranian": "zachodniopomorskie", "Zachodniopomorskie": "zachodniopomorskie",
    "Greater Poland": "wielkopolskie", "Wielkopolskie": "wielkopolskie",
    "Warmian-Masurian": "warmińsko-mazurskie", "Warmińsko-Mazurskie": "warmińsko-mazurskie",
    "Holy Cross": "świętokrzyskie", "Świętokrzyskie": "świętokrzyskie",
    "Silesian": "śląskie", "Śląskie": "śląskie",
    "Pomeranian": "pomorskie", "Pomorskie": "pomorskie",
    "Podlaskie": "podlaskie", 
    "Subcarpathian": "podkarpackie", "Podkarpackie": "podkarpackie",
    "Opole": "opolskie", "Opolskie": "opolskie",
    "Masovian": "mazowieckie", "Mazowieckie": "mazowieckie",
    "Lesser Poland": "małopolskie", "Małopolskie": "małopolskie",
    "Lubusz": "lubuskie", "Lubuskie": "lubuskie",
    "Lublin": "lubelskie", "Lubelskie": "lubelskie",
    "Łódź": "łódzkie", "Łódzkie": "łódzkie",
    "Kuyavian-Pomeranian": "kujawsko-pomorskie", "Kujawsko-Pomorskie": "kujawsko-pomorskie",
    "Lower Silesian": "dolnośląskie", "Dolnośląskie": "dolnośląskie"
}

# Создаем новую колонку специально для маппинга с GeoJSON. 
# Если значения нет в словаре (например, оно уже в нижнем регистре), переводим его в lower()
df_pd['geo_voivodeship'] = df_pd['voivodeship'].map(geojson_mapping).fillna(df_pd['voivodeship'].str.lower())

# Загрузка GeoJSON
url = "https://raw.githubusercontent.com/ppatrzyk/polska-geojson/master/wojewodztwa/wojewodztwa-min.geojson"
with urllib.request.urlopen(url) as response:
    poland_geojson = json.load(response)

# Построение Choropleth карты
fig = px.choropleth_mapbox(
    df_pd,
    geojson=poland_geojson,
    locations='geo_voivodeship',       # Изменено на подготовленную колонку
    featureidkey='properties.nazwa',   
    color='normalized_risk_index',
    color_continuous_scale='Reds',
    range_color=[df_pd['normalized_risk_index'].min()-0.5, df_pd['normalized_risk_index'].max()],
    mapbox_style="carto-positron",
    zoom=4.8,
    center={"lat": 52.0693, "lon": 19.4803},
    opacity=0.7,
    animation_frame='time_str',        
    labels={'normalized_risk_index': 'Hazard Risk'}
)

fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.show()